# Day 13 — Review and consolidate Days 1-12

One clean notebook that runs from top to bottom: explore, split, check for leakage, a regression model, a classification model with a threshold, and one honest test of each. Run each cell with **Shift + Enter**. `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

In [ ]:
# Get the dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-013/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd

# 1. Explore (Days 2-6): shape, missing values, summary statistics, correlations
df = pd.read_csv("student_data_150.csv")
df["AtRisk"] = (~df["Passed"]).astype(int)
features = ["StudyHours", "Attendance", "Age"]
print(f"Rows {len(df)} | columns {df.shape[1]} | missing values {df.isna().sum().sum()} | synthetic {(df['Source'] != 'original').sum()}")
print(df[features + ["ExamScore"]].describe().loc[["mean", "50%", "std"]].round(1).to_string())
print("Correlation with ExamScore:", df[features].corrwith(df["ExamScore"]).round(2).to_dict())

Rows 150 | columns 10 | missing values 0 | synthetic 142
      StudyHours  Attendance   Age  ExamScore
mean         3.6        79.5  11.0       57.4
50%          3.5        80.0  11.0       57.0
std          1.4        10.0   0.8       13.9
Correlation with ExamScore: {'StudyHours': 0.61, 'Attendance': 0.38, 'Age': 0.08}


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score

# 2. Split once (Day 8), stratified on the class, and check for leakage before modelling
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["AtRisk"])
print(f"Train+validation {len(train_val)} | test {len(test)} | students in both: {len(set(train_val['Student']) & set(test['Student']))}")

# Target leakage: ExamScore DEFINES at risk (below 50), so using it as a feature is cheating
leaky = LogisticRegression(max_iter=1000).fit(train_val[["ExamScore"]], train_val["AtRisk"])
print(f"With ExamScore as a feature: test accuracy {accuracy_score(test['AtRisk'], leaky.predict(test[['ExamScore']])):.2f}"
      " -- perfect, and useless: we never know the exam score in advance")

Train+validation 127 | test 23 | students in both: 0
With ExamScore as a feature: test accuracy 1.00 -- perfect, and useless: we never know the exam score in advance


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 3. Regression (Days 10-11): predict the exam SCORE. 5-fold CV on the 127, baseline first
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for name, m in [("Mean baseline", DummyRegressor()), ("Linear regression", LinearRegression())]:
    cv = cross_validate(m, train_val[features], train_val["ExamScore"], cv=kf,
                        scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"))
    print(f"{name:<18} CV MAE {-cv['test_neg_mean_absolute_error'].mean():5.2f} | "
          f"RMSE {-cv['test_neg_root_mean_squared_error'].mean():5.2f} | R2 {cv['test_r2'].mean():6.3f}")
reg = LinearRegression().fit(train_val[features], train_val["ExamScore"])
score_pred = reg.predict(test[features])
print(f"Test, once: MAE {mean_absolute_error(test['ExamScore'], score_pred):.2f} | "
      f"RMSE {np.sqrt(mean_squared_error(test['ExamScore'], score_pred)):.2f} | R2 {r2_score(test['ExamScore'], score_pred):.3f}")

Mean baseline      CV MAE 11.17 | RMSE 13.95 | R2 -0.063
Linear regression  CV MAE  8.81 | RMSE 10.96 | R2  0.317
Test, once: MAE 8.57 | RMSE 10.50 | R2 0.320


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 4. Classification (Day 12): predict the CLASS at risk. Thresholds on out-of-fold probabilities, never on the test set
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_tv = train_val["AtRisk"].values
oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), train_val[features], y_tv,
                        cv=skf, method="predict_proba")[:, 1]
print(f"Out-of-fold ROC-AUC {roc_auc_score(y_tv, oof):.3f}")
for t in [0.3, 0.5, 0.7]:
    pred = (oof >= t).astype(int)
    print(f"t = {t}: precision {precision_score(y_tv, pred):.2f} | recall {recall_score(y_tv, pred):.2f} | F1 {f1_score(y_tv, pred):.2f}")

Out-of-fold ROC-AUC 0.828
t = 0.3: precision 0.53 | recall 0.73 | F1 0.61
t = 0.5: precision 0.71 | recall 0.54 | F1 0.62
t = 0.7: precision 0.80 | recall 0.22 | F1 0.34


In [ ]:
# 5. Recommend 0.3 (goal fixed in advance: catch at-risk students; recall 0.73 against 0.54 at 0.5), then test ONCE
clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(train_val[features], y_tv)
p_test = clf.predict_proba(test[features])[:, 1]
pred = (p_test >= 0.3).astype(int)
y_te = test["AtRisk"].values
print(f"Test, once, t = 0.3: caught {((pred == 1) & (y_te == 1)).sum()} of {y_te.sum()} at risk | false alarms {((pred == 1) & (y_te == 0)).sum()} | "
      f"precision {precision_score(y_te, pred):.2f} | recall {recall_score(y_te, pred):.2f} | ROC-AUC {roc_auc_score(y_te, p_test):.3f}")

Test, once, t = 0.3: caught 4 of 7 at risk | false alarms 4 | precision 0.50 | recall 0.57 | ROC-AUC 0.701


In [ ]:
# 6. Regression vs classification on the SAME test students: a score in marks, and a probability of the class
both = test[["Student", "ExamScore", "AtRisk"]].assign(PredictedScore=score_pred.round(1), P_at_risk=p_test.round(2))
print(both.sort_values("P_at_risk", ascending=False).head(5).to_string(index=False))
agree = ((score_pred < 50) == (p_test >= 0.5)).mean()
print(f"The two models agree on 'below 50' vs 'at risk at 0.5' for {agree:.0%} of test students")

Student  ExamScore  AtRisk  PredictedScore  P_at_risk
   G002       30.0       1            35.0       0.93
   G024       62.0       0            45.2       0.71
   G139       53.0       0            46.7       0.63
   G015       53.0       0            48.0       0.58
   G010       34.0       1            50.9       0.50
The two models agree on 'below 50' vs 'at risk at 0.5' for 91% of test students
